In [4]:
import threading
import time
rlock = threading.RLock()
def recursive_task(depth:int):
    if depth <= 0:
        return
    with rlock:    
        print(f"  🔒 深度 {depth}: 获取锁成功")
        time.sleep(0.05)
        recursive_task(depth-1)
        print(f"  🔓 深度 {depth}: 释放锁")
if __name__ == "__main__":
    t = threading.Thread(target=recursive_task,args=(3,))
    t.start()
    t.join()

  🔒 深度 3: 获取锁成功
  🔒 深度 2: 获取锁成功
  🔒 深度 1: 获取锁成功
  🔓 深度 1: 释放锁
  🔓 深度 2: 释放锁
  🔓 深度 3: 释放锁


In [8]:
import threading
import time
data_ready = threading.Event()
shared_data = None
def producer():
    global shared_data
    print("  🏭 生产者: 正在准备数据...")
    time.sleep(2)
    shared_data = {'items':[1,2,3],'status':'ready'}
    print("  🏭 生产者: 数据准备完成！发出信号")
    data_ready.set()
def consumer():
    print("  👤 消费者: 等待数据就绪...")
    data_ready.wait()
    print(f"  👤 消费者: 收到信号！处理数据: {shared_data}")
if __name__ == '__main__':
    print("=== Event 事件通知演示 ===\n")
    t_producer = threading.Thread(target=producer)
    t_consumer = threading.Thread(target=consumer)
    t_consumer.start()
    t_producer.start()
    t_producer.join()
    t_consumer.join()
    print("\n🎉 生产-消费流程完成")

=== Event 事件通知演示 ===

  👤 消费者: 等待数据就绪...
  🏭 生产者: 正在准备数据...
  🏭 生产者: 数据准备完成！发出信号
  👤 消费者: 收到信号！处理数据: {'items': [1, 2, 3], 'status': 'ready'}

🎉 生产-消费流程完成


In [10]:
import threading
import time
import random
buffer = []
MAX_SIZE = 3
condition = threading.Condition()
def producer(pid:int,num_items:int):
    for i in range(num_items):
        with condition:
            while len(buffer) >= MAX_SIZE:
                print(f"  📦 生产者{pid}: 缓冲区满({len(buffer)}), 等待...")
                condition.wait()
            item = f"P{pid}_item{i}"
            buffer.append(item)
            print(f"  📥 生产者{pid}: 放入 {item} (缓冲: {len(buffer)})")
            condition.notify_all()
        time.sleep(random.uniform(0.1,0.3))
def consumer(cid:int,num_items:int):
    for _ in range(num_items):
        with condition:
            while len(buffer) == 0:
                print(f"  🛒 消费者{cid}: 缓冲区空, 等待...")
                condition.wait()
            item = buffer.pop(0)
            print(f"  📤 消费者{cid}: 取出 {item} (缓冲: {len(buffer)})")
            condition.notify_all()
        time.sleep(random.uniform(0.2,0.5))
if __name__ == "__main__":
    print("=== Condition 有界缓冲区演示 ===\n")
    threads = [
        threading.Thread(target=producer,args=(1,5)),
        threading.Thread(target=producer,args=(2,5)),
        threading.Thread(target=consumer,args=(1,5)),
        threading.Thread(target=consumer,args=(2,5)),
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    print(f"\n🎉 完成！最终缓冲区: {buffer}")

=== Condition 有界缓冲区演示 ===

  📥 生产者1: 放入 P1_item0 (缓冲: 1)
  📥 生产者2: 放入 P2_item0 (缓冲: 2)
  📤 消费者1: 取出 P1_item0 (缓冲: 1)
  📤 消费者2: 取出 P2_item0 (缓冲: 0)
  📥 生产者2: 放入 P2_item1 (缓冲: 1)
  📥 生产者1: 放入 P1_item1 (缓冲: 2)
  📤 消费者1: 取出 P2_item1 (缓冲: 1)
  📥 生产者2: 放入 P2_item2 (缓冲: 2)
  📥 生产者2: 放入 P2_item3 (缓冲: 3)
  📤 消费者2: 取出 P1_item1 (缓冲: 2)
  📥 生产者1: 放入 P1_item2 (缓冲: 3)
  📤 消费者1: 取出 P2_item2 (缓冲: 2)
  📥 生产者1: 放入 P1_item3 (缓冲: 3)
  📦 生产者2: 缓冲区满(3), 等待...
  📤 消费者2: 取出 P2_item3 (缓冲: 2)
  📥 生产者2: 放入 P2_item4 (缓冲: 3)
  📦 生产者1: 缓冲区满(3), 等待...
  📤 消费者1: 取出 P1_item2 (缓冲: 2)
  📥 生产者1: 放入 P1_item4 (缓冲: 3)
  📤 消费者2: 取出 P1_item3 (缓冲: 2)
  📤 消费者1: 取出 P2_item4 (缓冲: 1)
  📤 消费者2: 取出 P1_item4 (缓冲: 0)

🎉 完成！最终缓冲区: []


In [11]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
def fetch_page(url: str) -> dict:
    print(f"  ⏳ 抓取: {url}")
    time.sleep(1)  
    return {"url": url, "size": len(url) * 100, "status": 200}
if __name__ == '__main__':
    urls = [
        "https://example.com/page1",
        "https://example.com/page2",
        "https://example.com/page3",
        "https://example.com/page4",
        "https://example.com/page5",
    ]
    print("=== ThreadPoolExecutor 演示 ===\n")
    start = time.time()
    with ThreadPoolExecutor(max_workers=3) as executor:
        future_to_url = {executor.submit(fetch_page, url): url for url in urls}
        for future in as_completed(future_to_url):
            url = future_to_url[future]
            try:
                result = future.result(timeout=5)
                print(f"  ✅ {url} → size={result['size']}")
            except Exception as e:
                print(f"  ❌ {url} → 错误: {e}")

    elapsed = time.time() - start
    print(f"\n📊 总耗时: {elapsed:.2f} 秒 (5个URL, 3线程, 每个1s)")


=== ThreadPoolExecutor 演示 ===

  ⏳ 抓取: https://example.com/page1
  ⏳ 抓取: https://example.com/page2
  ⏳ 抓取: https://example.com/page3
  ⏳ 抓取: https://example.com/page4  ✅ https://example.com/page1 → size=2500
  ⏳ 抓取: https://example.com/page5
  ✅ https://example.com/page2 → size=2500
  ✅ https://example.com/page3 → size=2500

  ✅ https://example.com/page4 → size=2500
  ✅ https://example.com/page5 → size=2500

📊 总耗时: 2.00 秒 (5个URL, 3线程, 每个1s)
